<a href="https://colab.research.google.com/github/vadimdddd/CV_task_surface_damage_NN/blob/main/JPNotebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### pytorch-unet

https://github.com/usuyama/pytorch-unet

In [ ]:
import os

if not os.path.exists("pytorch_unet.py"):
  if not os.path.exists("pytorch_unet"):
    !git clone https://github.com/usuyama/pytorch-unet.git

  %cd pytorch-unet

##Required modules

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import torch
import torchvision.utils
import torch.nn as nn
import torchvision.models
import pytorch_unet
import torch.nn.functional as F
import torch
import torch.optim as optim
import time
import math

from collections import defaultdict
from loss import dice_loss
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets, models
from torchsummary import summary
from torch.optim import lr_scheduler

##Get dataset

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
!cp -r /content/gdrive/MyDrive/fishki_voc_data/ .

## Enabling GPU on Colab

Need to enable GPU from Notebook settings

- Navigate to Edit-Notebook settings menu
- Select GPU from the Hardware Accelerator dropdown list

In [ ]:
if not torch.cuda.is_available():
  raise Exception("GPU not availalbe. CPU training will be too slow.")

print("device name", torch.cuda.get_device_name(0))

##Prepare dataset

Train, validation, test split

In [ ]:
input_data_path = '/content/pytorch-unet/fishki_voc_data/JPEGImages/'
labels_path = '/content/pytorch-unet/fishki_voc_data/SegmentationClass/'

def read_files_in_folder(folder_path):
    file_list = []
    for file_name in sorted(os.listdir(folder_path)):
        file_path = os.path.join(folder_path, file_name)
        if os.path.isfile(file_path):
            file_list.append(file_path)
    return file_list

input_dataset = read_files_in_folder(input_data_path)
labels = read_files_in_folder(labels_path)
print(input_dataset)
print(labels)

In [ ]:
split_info_path = '/content/pytorch-unet/fishki_voc_data/ImageSets/Segmentation/'

def sort_numbers_in_files(folder_path):
    #train_samples, val_samples, test_samples = [], [], []

    for file_name in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file_name)

        if os.path.isfile(file_path):
            with open(file_path, 'r') as file:
                numbers = [line.strip() for line in file.readlines()]
                if file_name == 'train.txt':
                    train_samples = numbers
                elif file_name == 'test.txt':
                    test_samples = numbers
                elif file_name == 'val.txt':
                    val_samples = numbers

    return train_samples, test_samples, val_samples

train_samples, test_samples, val_samples = sort_numbers_in_files(split_info_path)
print(train_samples)
print(test_samples)
print(val_samples)

In [ ]:
def select_samples(input_dataset, labels, train_samples, test_samples, val_samples):
    input_dataset_train = []
    input_dataset_test = []
    input_dataset_val = []
    label_train = []
    label_test = []
    label_val = []

    for sample in train_samples:
        for i in range(len(input_dataset)):
            if sample in input_dataset[i]:
                input_dataset_train.append(input_dataset[i])
                label_train.append(labels[i])
                break

    for sample in test_samples:
        for i in range(len(input_dataset)):
            if sample in input_dataset[i]:
                input_dataset_test.append(input_dataset[i])
                label_test.append(labels[i])
                break

    for sample in val_samples:
        for i in range(len(input_dataset)):
            if sample in input_dataset[i]:
                input_dataset_val.append(input_dataset[i])
                label_val.append(labels[i])
                break

    return input_dataset_train, input_dataset_test, input_dataset_val, label_train, label_test, label_val

input_dataset_train, input_dataset_test, input_dataset_val, label_train, label_test, label_val = select_samples(input_dataset, labels, train_samples, test_samples, val_samples)
print(input_dataset_train[2])
print(input_dataset_test[10])
print(input_dataset_val[6])
print(label_train[2])
print(label_test[10])
print(label_val[6])

In [ ]:
def image_handler(image_list, image_type):
    handler_result = np.array([], dtype='float32')
    for image_path in image_list:
        image = cv2.imread(image_path)
        resized_image = cv2.resize(image, (192, 192))
        resized_array = np.array(resized_image)
        handler_result = np.append(handler_result, resized_array)
    return handler_result.reshape(-1, 192, 192, 3)

train_data = image_handler(input_dataset_train, 'jpg')
train_mask = image_handler(label_train, 'png')
train_mask = train_mask.astype(np.float32) / 128.0
test_data = image_handler(input_dataset_test, 'jpg')
test_mask = image_handler(label_test, 'png')
test_mask = test_mask.astype(np.float32) / 128.0
val_data = image_handler(input_dataset_val, 'jpg')
val_mask = image_handler(label_val, 'png')
val_mask = val_mask.astype(np.float32) / 128.0

In [ ]:
print(train_data.shape, train_data.min(), train_data.max())
print(train_mask.shape, train_mask.min(), train_mask.max())
print(test_data.shape, test_data.min(), test_data.max())
print(test_mask.shape, test_mask.min(), test_mask.max())
print(val_data.shape, val_data.min(), val_data.max())
print(val_mask.shape, val_mask.min(), val_mask.max())

def show_imgs():
  images_list = [54, 2, -1, 96]
  for img in images_list:
    image_to_view = train_data[img]
    plt.imshow((image_to_view * 255).astype(np.uint8))
    plt.axis('off')
    plt.show()

    image_to_view = train_mask[img]
    plt.imshow((image_to_view * 255).astype(np.uint8))
    plt.axis('off')
    plt.show()

show_imgs()

Preparing DataLoader

In [ ]:
class SimDataset(Dataset):
  def __init__(self, dataset, labels, transform=None):
    self.input_images, self.target_masks = dataset, labels
    self.transform = transform

  def __len__(self):
    return len(self.input_images)

  def __getitem__(self, idx):
    image = self.input_images[idx]
    mask = self.target_masks[idx]
    if self.transform:
      image = self.transform(image)

    return [image, mask]

# use the same transformations for train/val in this example
trans = transforms.Compose([
  transforms.ToTensor(),
  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # imagenet
])

train = SimDataset(train_data, train_mask, transform = trans)
val = SimDataset(val_data, val_mask, transform = trans)

image_datasets = {
  'train': train, 'val': val
}

batch_size = 25

dataloaders = {
  'train': DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=0),
  'val': DataLoader(val, batch_size=batch_size, shuffle=True, num_workers=0)
}

Check DataLoader output

In [ ]:
def reverse_transform(inp):
  inp = inp.numpy().transpose((1, 2, 0))
  mean = np.array([0.485, 0.456, 0.406])
  std = np.array([0.229, 0.224, 0.225])
  inp = std * inp + mean
  inp = (inp * 255).astype(np.uint8)

  return inp

# Get a batch of training data
inputs, masks = next(iter(dataloaders['train']))
masks = torch.transpose(masks, 1, -1)
print(inputs.shape, masks.shape)

plt.imshow(reverse_transform(inputs[4]))

## Define a UNet module

In [ ]:
def convrelu(in_channels, out_channels, kernel, padding):
  return nn.Sequential(
    nn.Conv2d(in_channels, out_channels, kernel, padding=padding),
    nn.ReLU(inplace=True),
  )


class ResNetUNet(nn.Module):
  def __init__(self, n_class):
    super().__init__()

    self.base_model = torchvision.models.resnet18(pretrained=True)
    self.base_layers = list(self.base_model.children())

    self.layer0 = nn.Sequential(*self.base_layers[:3]) # size=(N, 64, x.H/2, x.W/2)
    self.layer0_1x1 = convrelu(64, 64, 1, 0)
    self.layer1 = nn.Sequential(*self.base_layers[3:5]) # size=(N, 64, x.H/4, x.W/4)
    self.layer1_1x1 = convrelu(64, 64, 1, 0)
    self.layer2 = self.base_layers[5]  # size=(N, 128, x.H/8, x.W/8)
    self.layer2_1x1 = convrelu(128, 128, 1, 0)
    self.layer3 = self.base_layers[6]  # size=(N, 256, x.H/16, x.W/16)
    self.layer3_1x1 = convrelu(256, 256, 1, 0)
    self.layer4 = self.base_layers[7]  # size=(N, 512, x.H/32, x.W/32)
    self.layer4_1x1 = convrelu(512, 512, 1, 0)

    self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    self.conv_up3 = convrelu(256 + 512, 512, 3, 1)
    self.conv_up2 = convrelu(128 + 512, 256, 3, 1)
    self.conv_up1 = convrelu(64 + 256, 256, 3, 1)
    self.conv_up0 = convrelu(64 + 256, 128, 3, 1)

    self.conv_original_size0 = convrelu(3, 64, 3, 1)
    self.conv_original_size1 = convrelu(64, 64, 3, 1)
    self.conv_original_size2 = convrelu(64 + 128, 64, 3, 1)

    self.conv_last = nn.Conv2d(64, n_class, 1)

  def forward(self, input):
    x_original = self.conv_original_size0(input)
    x_original = self.conv_original_size1(x_original)

    layer0 = self.layer0(input)
    layer1 = self.layer1(layer0)
    layer2 = self.layer2(layer1)
    layer3 = self.layer3(layer2)
    layer4 = self.layer4(layer3)

    layer4 = self.layer4_1x1(layer4)
    x = self.upsample(layer4)
    layer3 = self.layer3_1x1(layer3)
    x = torch.cat([x, layer3], dim=1)
    x = self.conv_up3(x)

    x = self.upsample(x)
    layer2 = self.layer2_1x1(layer2)
    x = torch.cat([x, layer2], dim=1)
    x = self.conv_up2(x)

    x = self.upsample(x)
    layer1 = self.layer1_1x1(layer1)
    x = torch.cat([x, layer1], dim=1)
    x = self.conv_up1(x)

    x = self.upsample(x)
    layer0 = self.layer0_1x1(layer0)
    x = torch.cat([x, layer0], dim=1)
    x = self.conv_up0(x)

    x = self.upsample(x)
    x = torch.cat([x, x_original], dim=1)
    x = self.conv_original_size2(x)

    out = self.conv_last(x)

    return out

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)

model = ResNetUNet(6)
model = model.to(device)

In [ ]:
model

In [ ]:
summary(model, input_size=(3, 224, 224))

In [ ]:
checkpoint_path = "checkpoint.pth"

def calc_loss(pred, target, metrics, bce_weight=0.5):
    bce = F.binary_cross_entropy_with_logits(pred, target)

    pred = torch.sigmoid(pred)
    miou = dice_loss(pred, target)

    loss = bce * bce_weight + miou * (1 - bce_weight)

    metrics['bce'] += bce.data.cpu().numpy() * target.size(0)
    metrics['miou'] += miou.data.cpu().numpy() * target.size(0)
    metrics['loss'] += loss.data.cpu().numpy() * target.size(0)

    return loss

def print_metrics(metrics, epoch_samples, phase):
    outputs = []
    for k in metrics.keys():
        outputs.append("{}: {:4f}".format(k, metrics[k] / epoch_samples))

    print("{}: {}".format(phase, ", ".join(outputs)))

def train_model(model, optimizer, scheduler, num_epochs=25):
    best_loss = 1e10

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        since = time.time()

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            metrics = defaultdict(float)
            epoch_samples = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = torch.transpose(labels, 1, -1)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = calc_loss(outputs, labels, metrics)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                epoch_samples += inputs.size(0)

            print_metrics(metrics, epoch_samples, phase)
            epoch_loss = metrics['loss'] / epoch_samples

            if phase == 'train':
              scheduler.step()
              for param_group in optimizer.param_groups:
                  print("LR", param_group['lr'])

            # save the model weights
            if phase == 'val' and epoch_loss < best_loss:
                print(f"saving best model to {checkpoint_path}")
                best_loss = epoch_loss
                torch.save(model.state_dict(), checkpoint_path)

        time_elapsed = time.time() - since
        print('{:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))

    print('Best val loss: {:4f}'.format(best_loss))

    # load best model weights
    model.load_state_dict(torch.load(checkpoint_path))
    return model

In [ ]:
num_class = 3
model = ResNetUNet(num_class).to(device)

# freeze backbone layers
for l in model.base_layers:
  for param in l.parameters():
    param.requires_grad = False

optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=8, gamma=0.1)

model = train_model(model, optimizer_ft, exp_lr_scheduler, num_epochs=100)

In [ ]:
model.eval()   # Set model to the evaluation mode

# Create a new simulation dataset for testing
test_dataset = SimDataset(test_data, test_mask, transform = trans)
test_loader = DataLoader(test_dataset, batch_size=14, shuffle=False, num_workers=0)

# Get the first batch
inputs, labels = next(iter(test_loader))
inputs = inputs.to(device)
labels = labels.to(device)
labels = torch.transpose(labels, 2, 3)
labels = torch.transpose(labels, 1, 2)

print('inputs.shape', inputs.shape)
print('labels.shape', labels.shape)

# Predict
pred = model(inputs)
# The loss functions include the sigmoid function.
pred = torch.sigmoid(pred)
pred = pred.data.cpu().numpy()
pred = torch.from_numpy(pred)
print('pred.shape', pred.shape)

# Show results for input_image/output_image/predicted_output_image
def display_images(images):
    fig, axs = plt.subplots(1, len(images))
    for i, image in enumerate(images):
        if image.is_cuda:
            image = image.cpu()
        image = image.detach().numpy()
        axs[i].imshow(image.transpose(1, 2, 0))
        axs[i].axis('off')
    plt.show()

test_images = [0, 1, 3, 11]
for i in test_images:
#i = -3
  input_pic = (inputs[i] - inputs[i].min()) / (inputs[i].max() - inputs[i].min())
  images_to_display = [input_pic, labels[i], pred[i]]
  display_images(images_to_display)